# Nuclear Waste Canister Temperature Prediction — KNN
**CIVIL-226 - Introduction to Machine Learning for Engineers — EPFL**  
**Team name:** [To fill]  
**Members:** NAJA Nour | CLERICI Alessandro

## Objective
Predict the temperature around nuclear waste canisters at unobserved sensor positions using k-Nearest Neighbors regression.

The dataset contains 3 experimental replications per sensor/timestep (different power scenarios). The test set corresponds exclusively to Rep 2 (radioactive decay profile), so we train only on Rep 2 data for consistency.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
print('Imports OK')

## 2. Load data

In [ ]:
train   = pd.read_parquet('data/train.parquet')
test    = pd.read_parquet('data/test.parquet')
sensors = pd.read_parquet('data/sensors.parquet')

# Remove duplicate sensors (N206, N213 appear twice with identical coordinates)
n_before = len(sensors)
sensors = sensors.drop_duplicates(subset='sensor', keep='first').reset_index(drop=True)
print(f'Duplicate sensors removed: {n_before - len(sensors)}')

print(f'Sensors : {sensors.shape}  ->  {sensors.columns.tolist()}')
print(f'Train   : {train.shape}   ->  {train.columns.tolist()}')
print(f'Test    : {test.shape}    ->  {test.columns.tolist()}')

## 3. Merge sensor positions

In [ ]:
train_full = train.merge(sensors, on='sensor', how='left')
print(train_full.head())
print(train_full.shape)

## 4. Clean missing values

In [ ]:
print(f'Rows before NaN removal: {len(train_full)}')
train_full = train_full.dropna(subset=['temperature']).copy()
print(f'Rows after NaN removal : {len(train_full)}')

## 5. Data Exploration

In [ ]:
print(train_full['temperature'].describe(percentiles=[0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999]))
print('Temperature min:', train_full['temperature'].min())
print('Temperature max:', train_full['temperature'].max())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train_full['temperature'].dropna(), bins=100, color='steelblue', edgecolor='white')
axes[0].set_title('Temperature distribution')
axes[0].set_xlabel('Temperature [°C]')

temp_by_time = train_full.groupby('time')['temperature'].mean()
axes[1].plot(temp_by_time.index / (365.25*24*3600), temp_by_time.values, color='tomato', linewidth=0.8)
axes[1].set_title('Mean temperature over time')
axes[1].set_xlabel('Time [years]')
axes[1].set_ylabel('Mean temperature [°C]')
plt.tight_layout()
plt.show()

## 6. Outlier Removal

We use two complementary methods from OG_improved:
- **Global MAD** (Median Absolute Deviation): catches sensors with physically impossible temperatures
- **Local IQR per timestep**: catches values that deviate too much from the distribution at a given moment in time

In [ ]:
# --- Global outliers via MAD ---
temp = train_full['temperature']
median_temp = temp.median()
mad_temp = np.median(np.abs(temp - median_temp))
robust_z = 0.6745 * (temp - median_temp) / (mad_temp + 1e-8)

train_full['is_global_outlier'] = robust_z.abs() > 6
print(f'Global outliers (MAD): {train_full["is_global_outlier"].sum()}')

In [ ]:
# --- Local outliers per timestep via IQR ---
time_stats = (
    train_full
    .groupby('time')['temperature']
    .agg(
        time_median='median',
        time_q1=lambda x: x.quantile(0.25),
        time_q3=lambda x: x.quantile(0.75)
    )
    .reset_index()
)
time_stats['time_iqr'] = time_stats['time_q3'] - time_stats['time_q1']
train_full = train_full.merge(time_stats, on='time', how='left')

train_full['is_local_outlier'] = (
    (train_full['temperature'] < train_full['time_median'] - 4.0 * train_full['time_iqr']) |
    (train_full['temperature'] > train_full['time_median'] + 4.0 * train_full['time_iqr'])
)
print(f'Local outliers (IQR per timestep): {train_full["is_local_outlier"].sum()}')

In [ ]:
# --- Remove outliers ---
before_rows = len(train_full)
train_full = train_full[
    (~train_full['is_global_outlier']) & (~train_full['is_local_outlier'])
].copy()
print(f'Rows removed: {before_rows - len(train_full)} | Remaining: {len(train_full)}')

# Drop temporary diagnostic columns
cols_to_drop = ['is_global_outlier', 'is_local_outlier',
                'time_median', 'time_q1', 'time_q3', 'time_iqr']
train_full = train_full.drop(columns=[c for c in cols_to_drop if c in train_full.columns])

## 7. Sensor Drift Detection & Correction

Some sensors show a systematic linear drift in their residuals over time — i.e. they read progressively higher or lower relative to the global trend. We detect and correct this.

In [ ]:
# Global median temperature per timestep
time_median = (
    train_full.groupby('time')['temperature']
    .median()
    .rename('global_time_median')
    .reset_index()
)
drift_df = train_full.merge(time_median, on='time', how='left').copy()
drift_df['temp_residual'] = drift_df['temperature'] - drift_df['global_time_median']

t_min = drift_df['time'].min()
t_max = drift_df['time'].max()
drift_df['time_norm_for_drift'] = (drift_df['time'] - t_min) / (t_max - t_min + 1e-8)

drift_records = []
for sensor_id, g in drift_df.groupby('sensor'):
    g = g.sort_values('time_norm_for_drift')
    if len(g) < 20:
        continue
    x = g['time_norm_for_drift'].values
    y = g['temp_residual'].values
    slope = np.polyfit(x, y, 1)[0] if np.std(y) > 1e-8 else 0.0
    corr  = np.corrcoef(x, y)[0, 1] if np.std(y) > 1e-8 else 0.0
    drift_records.append({'sensor': sensor_id, 'drift_slope': slope, 'drift_corr': corr})

sensor_drift = pd.DataFrame(drift_records)
slope_abs = sensor_drift['drift_slope'].abs()
slope_threshold = slope_abs.median() + 4 * (np.median(np.abs(slope_abs - slope_abs.median())) + 1e-8)
sensor_drift['is_drift_sensor'] = (
    (sensor_drift['drift_slope'].abs() > slope_threshold) &
    (sensor_drift['drift_corr'].abs() > 0.5)
)
drift_sensors = sensor_drift.loc[sensor_drift['is_drift_sensor'], 'sensor'].unique()
print(f'Slope threshold: {slope_threshold:.4f}')
print(f'Drift sensors detected: {len(drift_sensors)}')

In [ ]:
# Apply drift correction
t_min = train_full['time'].min()
t_max = train_full['time'].max()
train_full['time_norm_for_drift'] = (train_full['time'] - t_min) / (t_max - t_min + 1e-8)
drift_map = sensor_drift.set_index('sensor')['drift_slope'].to_dict()
train_full['drift_correction'] = 0.0

for s in drift_sensors:
    mask = train_full['sensor'] == s
    train_full.loc[mask, 'drift_correction'] = (
        drift_map[s] * (train_full.loc[mask, 'time_norm_for_drift'] - 0.5)
    )

train_full['temperature'] = train_full['temperature'] - train_full['drift_correction']
train_full = train_full.drop(columns=['time_norm_for_drift', 'drift_correction'])
print('Drift correction applied.')

## 8. Identify Replication & Filter Rep 2

Each (sensor, timestep) has exactly 3 rows corresponding to 3 experimental replications with different power scenarios:
- **Rep 0**: starts at 500W, drops to 0W around year 100
- **Rep 1**: starts at 1400W, drops to 0W around year 110  
- **Rep 2**: radioactive decay 1488W → 299W over 250 years

The test set corresponds **exclusively to Rep 2**. We therefore train only on Rep 2 to avoid introducing noise from physically incompatible power scenarios.

In [ ]:
# Identify replication by power rank at each (sensor, time)
# Rep 2 = highest power (radioactive decay from ~1488W)
train_full['rep'] = (
    train_full.groupby(['sensor', 'time'])['power']
    .rank(method='first')
    .astype(int) - 1
)

for rep in [0, 1, 2]:
    sub = train_full[train_full['rep'] == rep]
    print(f'Rep {rep}: {sub.sensor.nunique()} sensors, {len(sub):,} rows, '
          f'power {sub.power.iloc[0]:.0f}W -> {sub.power.iloc[-1]:.0f}W')

# Keep only Rep 2
train_full = train_full[train_full['rep'] == 2].drop(columns=['rep']).copy()
print(f'\nAfter filtering to Rep 2: {len(train_full):,} rows, {train_full.sensor.nunique()} sensors')

## 9. Feature Engineering

We add physics-motivated features:
- **dist_center**: distance from origin (r = sqrt(x²+y²))
- **dist_canister**: distance from estimated canister center
- **is_opa**: zone indicator (buffer vs OPA, threshold at x=1.4m)
- **time_log**: log(1+t) captures the logarithmic thermal diffusion
- **cum_energy**: cumulative energy injected into the system
- **power_over_dist2**: power weighted by inverse square distance (heat flux proxy)
- **diffusion**: dist²/time (diffusion number)

In [ ]:
eps = 1e-8

def add_features(df, t_max_ref):
    """
    Add physics-motivated engineered features.
    t_max_ref: max time from training data (for consistent time_norm on test).
    """
    df = df.copy()

    # Spatial features
    df['dist2']        = df['coor_x']**2 + df['coor_y']**2
    df['dist_center']  = np.sqrt(df['dist2'])
    df['dist_canister'] = np.sqrt((df['coor_x'] - 0.7)**2 + (df['coor_y'] - 1.2)**2)
    df['is_opa']       = (df['coor_x'] > 1.4).astype(float)

    # Temporal features
    df['time_log']  = np.log1p(df['time'])
    df['time_norm'] = df['time'] / t_max_ref

    # Physics-motivated features
    df['power_over_dist2'] = df['power'] / (df['dist2'] + eps)
    df['diffusion']        = df['dist2'] / (df['time'] + eps)

    return df


def add_cum_energy(df):
    """
    Compute cumulative energy from power x dt, then merge back.
    Uses drop_duplicates on time to avoid duplication when multiple sensors
    share the same timestep.
    """
    power_time = (
        df[['time', 'power']]
        .drop_duplicates(subset=['time'])
        .sort_values('time')
        .copy()
    )
    power_time['dt']         = power_time['time'].diff().fillna(0)
    power_time['cum_energy'] = (power_time['power'] * power_time['dt']).cumsum()

    return df.merge(
        power_time[['time', 'cum_energy']].drop_duplicates(subset=['time']),
        on='time', how='left'
    )


t_max_ref = train_full['time'].max()

train_full = add_cum_energy(train_full)
train_full = add_features(train_full, t_max_ref)

print('Features after engineering:', train_full.columns.tolist())
print(train_full.shape)

## 10. Final Checks After Cleaning

In [ ]:
print('Final train_full shape:', train_full.shape)
print('Missing values:')
print(train_full.isna().sum())

assert train_full['temperature'].notna().all()
assert np.isfinite(train_full['temperature']).all()
print('All checks passed.')

## 11. Train/Validation Split by Sensor

We split by sensor (not by row) to properly evaluate generalization to unseen sensor positions — which is exactly the test scenario. A row-based split would leak spatial information.

In [ ]:
FEATURES = [
    'coor_x', 'coor_y',
    'power', 'time_log', 'time_norm',
    'cum_energy',
    'dist_center', 'dist_canister', 'is_opa',
    'power_over_dist2', 'diffusion'
]
TARGET = 'temperature'

unique_sensors = train_full['sensor'].unique()
rng = np.random.default_rng(42)
val_sensors   = rng.choice(unique_sensors, size=int(0.2 * len(unique_sensors)), replace=False)
train_sensors = np.setdiff1d(unique_sensors, val_sensors)

train_df = train_full[train_full['sensor'].isin(train_sensors)].copy()
val_df   = train_full[train_full['sensor'].isin(val_sensors)].copy()

assert set(train_df['sensor']).isdisjoint(set(val_df['sensor']))

print(f'Train sensors: {train_df.sensor.nunique()} | Rows: {len(train_df):,}')
print(f'Val   sensors: {val_df.sensor.nunique()}   | Rows: {len(val_df):,}')

## 12. Normalisation

**Critical for KNN**: distance is computed directly on feature values, so features with large scales (e.g. time in seconds ~10⁹) would completely dominate over spatial coordinates (~1-10m). StandardScaler ensures all features contribute equitably to the distance computation.

In [ ]:
X_train = train_df[FEATURES].values
y_train = train_df[TARGET].values

X_val   = val_df[FEATURES].values
y_val   = val_df[TARGET].values

# Fit scaler ONLY on training data, then apply to val and test
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)

print(f'X_train: {X_train_s.shape} | X_val: {X_val_s.shape}')
print(f'NaN in X_train_s: {np.isnan(X_train_s).sum()} | Inf: {np.isinf(X_train_s).sum()}')
print(f'NaN in X_val_s:   {np.isnan(X_val_s).sum()} | Inf: {np.isinf(X_val_s).sum()}')

## 13. Hyperparameter Tuning — Best k

We search for the best k on a subsample for speed, then train the final model on the full training set.

In [ ]:
# Subsample for fast k search
sample_size = min(100_000, len(X_train_s))
idx_sample  = np.random.choice(len(X_train_s), sample_size, replace=False)

k_values = [3, 5, 10, 20, 50]
rmse_k   = []

for k in k_values:
    knn = KNeighborsRegressor(n_neighbors=k, n_jobs=-1)
    knn.fit(X_train_s[idx_sample], y_train[idx_sample])
    pred = knn.predict(X_val_s[:10_000])  # partial val for speed
    rmse = np.sqrt(mean_squared_error(y_val[:10_000], pred))
    rmse_k.append(rmse)
    print(f'k={k:3d}  ->  RMSE = {rmse:.4f}')

best_k = k_values[np.argmin(rmse_k)]
print(f'\nBest k: {best_k}')

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(k_values, rmse_k, 'o-', color='steelblue')
plt.axvline(best_k, color='tomato', linestyle='--', label=f'Best k={best_k}')
plt.xlabel('k')
plt.ylabel('RMSE (validation sample)')
plt.title('Hyperparameter search: k selection')
plt.legend()
plt.tight_layout()
plt.show()

## 14. Final KNN Training & Evaluation

In [ ]:
# Train on full training set with best k
knn_final = KNeighborsRegressor(n_neighbors=best_k, n_jobs=-1)
knn_final.fit(X_train_s, y_train)
print(f'KNN trained with k={best_k} on {len(X_train_s):,} samples.')

In [ ]:
# Evaluate on full validation set
y_val_pred = knn_final.predict(X_val_s)

mae  = mean_absolute_error(y_val, y_val_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
r2   = r2_score(y_val, y_val_pred)

print(f'Validation MAE  : {mae:.4f} °C')
print(f'Validation RMSE : {rmse:.4f} °C')
print(f'Validation R²   : {r2:.4f}')

In [ ]:
# True vs predicted scatter plot
plt.figure(figsize=(6, 6))
plt.scatter(y_val, y_val_pred, s=3, alpha=0.3)
mn, mx = min(y_val.min(), y_val_pred.min()), max(y_val.max(), y_val_pred.max())
plt.plot([mn, mx], [mn, mx], 'r--')
plt.xlabel('True temperature [°C]')
plt.ylabel('Predicted temperature [°C]')
plt.title(f'KNN (k={best_k}) — True vs Predicted on unseen sensors')
plt.grid(True)
plt.tight_layout()
plt.show()

## 15. Error Analysis by Sensor

In [ ]:
val_results = val_df.copy()
val_results['y_true'] = y_val
val_results['y_pred'] = y_val_pred
val_results['abs_error'] = np.abs(y_val - y_val_pred)
val_results['sq_error']  = (y_val - y_val_pred)**2

sensor_metrics = (
    val_results
    .groupby('sensor')
    .agg(
        mae  =('abs_error', 'mean'),
        rmse =('sq_error',  lambda x: np.sqrt(np.mean(x))),
        coor_x=('coor_x', 'first'),
        coor_y=('coor_y', 'first'),
    )
    .sort_values('rmse', ascending=False)
)
print('Worst sensors by RMSE:')
print(sensor_metrics.head(10))

In [ ]:
# Spatial error map
plt.figure(figsize=(8, 5))
sc = plt.scatter(
    sensor_metrics['coor_x'],
    sensor_metrics['coor_y'],
    c=sensor_metrics['rmse'],
    cmap='hot_r', s=80
)
plt.colorbar(sc, label='RMSE [°C]')
plt.xlabel('coor_x')
plt.ylabel('coor_y')
plt.title('Validation RMSE by sensor position')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 16. Final Predictions & Submission

In [ ]:
# Build test_full with same preprocessing pipeline
test_full = test.merge(sensors, on='sensor', how='left')

# Cumulative energy on test (same logic as train)
test_full = add_cum_energy(test_full)

# Engineered features (using t_max_ref from train for consistent time_norm)
test_full = add_features(test_full, t_max_ref)

print(f'test_full shape: {test_full.shape}')
print(f'Missing values: {test_full[FEATURES].isna().sum().sum()}')

In [ ]:
# Normalize using the SAME scaler fitted on train
X_test_s = scaler.transform(test_full[FEATURES].values)

assert len(X_test_s) == len(test), 'Test size mismatch!'
print(f'X_test_s: {X_test_s.shape}')

In [ ]:
# Predict
y_pred = knn_final.predict(X_test_s)

# Build submission
submission = pd.DataFrame({
    'Id': np.arange(len(test), dtype=int),
    'temperature': y_pred.astype(float)
})

assert list(submission.columns) == ['Id', 'temperature']
assert len(submission) == len(test)
assert np.isfinite(submission['temperature']).all()
assert submission.isna().sum().sum() == 0

submission.to_csv('submission.csv', index=False)
print(f'submission.csv saved — {len(submission)} rows')
display(submission.head())

In [ ]:
# Visual sanity check: train vs test predictions over time
test_with_pred = test.copy()
test_with_pred['temperature_pred'] = y_pred

temp_train = train_full.groupby('time')['temperature'].mean()
temp_pred  = test_with_pred.groupby('time')['temperature_pred'].mean()

plt.figure(figsize=(10, 4))
plt.plot(temp_train.index / (365.25*24*3600), temp_train.values,
         label='Train Rep2 (true)', color='tomato', alpha=0.7, linewidth=0.8)
plt.plot(temp_pred.index / (365.25*24*3600), temp_pred.values,
         label='Test (predicted)', color='steelblue', alpha=0.7, linewidth=0.8)
plt.title('Sanity check: Train vs Test predictions over time')
plt.xlabel('Time [years]')
plt.ylabel('Mean temperature [°C]')
plt.legend()
plt.tight_layout()
plt.show()